# PCB Defect Detection Using YOLOv8
### Automated Visual Inspection of Printed Circuit Boards

**Dataset:** DeepPCB — 6 defect classes: `open`, `short`, `mousebite`, `spur`, `copper`, `pin-hole`  
**Model:** YOLOv8s (Ultralytics)  
**Runtime:** Google Colab with T4 GPU  

---

**Pipeline Overview:**
1. Install dependencies
2. Acquire the DeepPCB dataset
3. Convert annotations to YOLO format
4. Create train/validation split
5. Train YOLOv8s
6. Visualise inference results
7. Evaluate per-class metrics
8. Export trained model and results

---
## 1. Environment Setup
Install the Ultralytics library (includes YOLOv8) and supporting packages.  
**Action required:** Enable GPU — *Runtime → Change runtime type → T4 GPU* — before running this cell.

In [ ]:
!pip install ultralytics opencv-python-headless matplotlib --quiet

# Verify GPU is available
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

---
## 2. Dataset Acquisition
Clone the [DeepPCB repository](https://github.com/tangsanli5201/DeepPCB) from GitHub.  
It contains 1,500 PCB image pairs with six defect categories annotated in a custom format.

In [ ]:
import os

!git clone https://github.com/tangsanli5201/DeepPCB.git --quiet

if os.path.exists('DeepPCB'):
    print("✅ DeepPCB cloned successfully.")
    print("Contents:", os.listdir('DeepPCB'))
else:
    print("❌ Clone failed — check your internet connection.")

---
## 3. Annotation Conversion: DeepPCB → YOLO Format

**DeepPCB annotation format (per line):** `x1 y1 x2 y2 class_id`  
- Coordinates are absolute pixel values (top-left / bottom-right corners)
- Class IDs are 1-indexed (1 = open, 2 = short, … 6 = pin-hole)
- Annotations live in `groupXXXXX/<id>_not/` folders
- Images live in `groupXXXXX/<id>/` as `<id>_test.jpg` or `<id>_temp.jpg`

**YOLO format (per line):** `class_id cx cy w h` (all values normalised 0–1, 0-indexed classes)

In [ ]:
import os, glob, shutil
from PIL import Image

# DeepPCB uses 1-indexed classes; YOLO expects 0-indexed
CLASS_MAP  = {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5}
CLASS_NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']

def convert_annotation(txt_path: str, img_path: str, out_path: str) -> int:
    """Convert a single DeepPCB annotation file to YOLO format.
    Returns the number of valid bounding boxes written."""
    img   = Image.open(img_path)
    W, H  = img.size
    lines = []
    with open(txt_path) as f:
        for raw in f:
            parts = raw.strip().split()
            if len(parts) != 5:
                continue
            x1, y1, x2, y2, cls = parts
            if cls not in CLASS_MAP:
                continue
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            cx = ((x1 + x2) / 2) / W
            cy = ((y1 + y2) / 2) / H
            w  = (x2 - x1) / W
            h  = (y2 - y1) / H
            lines.append(f"{CLASS_MAP[cls]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines))
    return len(lines)

# Create output directories
os.makedirs('dataset/images', exist_ok=True)
os.makedirs('dataset/labels', exist_ok=True)

converted, skipped, total_boxes = 0, 0, 0

for txt in glob.glob('DeepPCB/PCBData/group*/[0-9]*_not/*.txt'):
    stem   = os.path.splitext(os.path.basename(txt))[0]   # e.g. 20085088
    imgdir = os.path.dirname(txt).replace('_not', '')      # sibling folder without _not

    # Images can be named either _test.jpg or _temp.jpg
    img_path = None
    for suffix in ['_test.jpg', '_temp.jpg']:
        candidate = os.path.join(imgdir, stem + suffix)
        if os.path.exists(candidate):
            img_path = candidate
            break

    if img_path is None:
        skipped += 1
        continue

    n = convert_annotation(txt, img_path, f'dataset/labels/{stem}.txt')
    shutil.copy(img_path, f'dataset/images/{stem}.jpg')
    total_boxes += n
    converted   += 1

print(f"✅ Converted  : {converted} samples")
print(f"⚠️  Skipped    : {skipped} (no matching image found)")
print(f"📦 Images     : {len(os.listdir('dataset/images'))}")
print(f"🏷️  Labels     : {len(os.listdir('dataset/labels'))}")
print(f"📐 Total boxes: {total_boxes}")

---
## 4. Train / Validation Split
Split the dataset 80 % train / 20 % validation with a fixed random seed for reproducibility.  
Then write the `pcb.yaml` configuration file that YOLOv8 needs to locate the data.

In [ ]:
import os, shutil, random

random.seed(42)  # Fixed seed — ensures reproducible splits across runs

all_imgs = os.listdir('dataset/images')
random.shuffle(all_imgs)

split_idx  = int(0.8 * len(all_imgs))
train_imgs = all_imgs[:split_idx]
val_imgs   = all_imgs[split_idx:]

for split_name, split_list in [('train', train_imgs), ('val', val_imgs)]:
    os.makedirs(f'dataset/{split_name}/images', exist_ok=True)
    os.makedirs(f'dataset/{split_name}/labels', exist_ok=True)
    for img in split_list:
        shutil.copy(f'dataset/images/{img}',
                    f'dataset/{split_name}/images/{img}')
        lbl = img.replace('.jpg', '.txt')
        lbl_src = f'dataset/labels/{lbl}'
        if os.path.exists(lbl_src):
            shutil.copy(lbl_src, f'dataset/{split_name}/labels/{lbl}')

yaml_content = """\
path: /content/dataset
train: train/images
val:   val/images

nc: 6
names: ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
"""
with open('pcb.yaml', 'w') as f:
    f.write(yaml_content)

print(f"Train : {len(train_imgs)} images  ({len(train_imgs)/len(all_imgs)*100:.0f}%)")
print(f"Val   : {len(val_imgs)} images  ({len(val_imgs)/len(all_imgs)*100:.0f}%)")
print("YAML  : pcb.yaml written ✅")

---
## 5. Model Training

**Architecture:** YOLOv8s (small variant) — a good balance of speed and accuracy for single-class-dense datasets like PCB.  
**Hyperparameters chosen:**
- `imgsz=640` — standard YOLOv8 input resolution
- `batch=16` — fits comfortably on T4 VRAM
- `epochs=50` with `patience=10` early stopping
- Pretrained ImageNet weights (`yolov8s.pt`) for transfer learning

Training time: approximately 25–35 minutes on T4 GPU.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # Downloads pretrained weights automatically

results = model.train(
    data    = '/content/pcb.yaml',
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    name    = 'pcb_defect_yolov8s',
    project = '/content/runs',
    patience= 10,       # Early stopping: stop if no improvement for 10 epochs
    verbose = False,    # Set True for epoch-by-epoch logs
    exist_ok= True      # Overwrite run folder if re-running this cell
)

print("\n✅ Training complete!")
print(f"Best model saved at: /content/runs/pcb_defect_yolov8s/weights/best.pt")

---
## 6. Inference Visualisation
Load the best checkpoint and run it on four random validation images to qualitatively inspect detection quality.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import matplotlib.pyplot as plt
import cv2, glob, random, numpy as np

MODEL_PATH = '/content/runs/pcb_defect_yolov8s/weights/best.pt'
model = YOLO(MODEL_PATH)

# Show training loss / metric curves
print("Training curves:")
display(IPImage('/content/runs/pcb_defect_yolov8s/results.png'))

# Run inference on 4 random validation images
random.seed(0)
test_imgs = random.sample(glob.glob('dataset/val/images/*.jpg'), 4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_path in zip(axes.flatten(), test_imgs):
    result   = model.predict(img_path, conf=0.25, verbose=False)[0]
    annotated = result.plot()  # Returns BGR numpy array with drawn boxes
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    n = len(result.boxes)
    ax.set_title(f'{n} defect(s) detected', fontsize=11, fontweight='bold')

plt.suptitle('PCB Defect Detection — YOLOv8s Inference Results',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('detection_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Saved: detection_results.png")

---
## 7. Quantitative Evaluation
Compute per-class Precision, Recall, mAP@0.5, and mAP@0.5:0.95 on the validation set.  
Copy this table directly into your research paper.

In [ ]:
metrics = model.val(data='/content/pcb.yaml', verbose=False)

classes = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']

print(f"\n{'Class':<12} {'Precision':>10} {'Recall':>8} {'mAP@0.5':>9} {'mAP@0.5:0.95':>13}")
print("─" * 56)
for i, cls in enumerate(classes):
    p      = metrics.box.p[i]
    r      = metrics.box.r[i]
    m50    = metrics.box.ap50[i]
    m5095  = metrics.box.ap[i]
    print(f"{cls:<12} {p:>10.3f} {r:>8.3f} {m50:>9.3f} {m5095:>13.3f}")
print("─" * 56)
print(f"{'ALL':<12} {metrics.box.mp:>10.3f} {metrics.box.mr:>8.3f} "
      f"{metrics.box.map50:>9.3f} {metrics.box.map:>13.3f}")
print("\n📋 Copy this table into your research paper.")

---
## 8. Export — Download Results
Download:
- `best.pt` — the trained model weights (keep this safe)
- `pcb_results.zip` — training curves, confusion matrix, precision-recall curves
- `detection_results.png` — sample inference grid

In [ ]:
from google.colab import files
import shutil

# Zip the full run folder (curves, confusion matrix, config)
shutil.make_archive('pcb_results', 'zip', '/content/runs/pcb_defect_yolov8s')

print("Downloading best.pt ...")
files.download('/content/runs/pcb_defect_yolov8s/weights/best.pt')

print("Downloading pcb_results.zip ...")
files.download('pcb_results.zip')

print("Downloading detection_results.png ...")
files.download('detection_results.png')

print("\n✅ All files downloaded. Store best.pt — it is your trained model.")